## Loading All 4 Feature Files

In [11]:

import pandas as pd
import numpy as np

df_billings = pd.read_csv('../../../data/interim/billings_features.csv')
df_renewal  = pd.read_csv('../../../data/interim/renewal_calls_features.csv')
df_email    = pd.read_csv('../../../data/interim/email_features.csv')
df_cc       = pd.read_csv('../../../data/interim/cc_calls_features.csv')

print(f"Billings : {df_billings.shape}")
print(f"Renewal Calls : {df_renewal.shape}")
print(f"Emails : {df_email.shape}")
print(f"CC Calls: {df_cc.shape}")
pd.set_option('display.max_columns', None)

Billings : (113769, 63)
Renewal Calls : (968, 37)
Emails : (21362, 32)
CC Calls: (14988, 45)



## Verifying co_ref is in All Files

In [12]:

for name, df in [('billings', df_billings), ('renewal', df_renewal),
                 ('email', df_email), ('cc', df_cc)]:
    has = 'co_ref' in df.columns
    unique = df['co_ref'].nunique() if has else 0
    print(f"{name:12s} — co_ref: {has} | unique customers: {unique:,}")

billings     — co_ref: True | unique customers: 46,328
renewal      — co_ref: True | unique customers: 968
email        — co_ref: True | unique customers: 21,362
cc           — co_ref: True | unique customers: 14,988


## Join All Datasets onto Billings

In [13]:
# Billings is the base — left join everything onto it
df = df_billings.copy()

df = df.merge(df_renewal, on='co_ref', how='left')
df = df.merge(df_email,   on='co_ref', how='left')
df = df.merge(df_cc,      on='co_ref', how='left')

print(f"Shape after join: {df.shape}")

Shape after join: (113769, 174)


In [14]:
print(df.columns.tolist())

['co_ref', 'renewal_month', 'sustainability_score', 'total_renewal_score_new', 'last_years_price', 'auto_renewal_score', 'status_scores', 'anchoring_score', 'tenure_scores', 'proforma_auto_renewal', 'proforma_world_pay_token', 'proforma_date', 'current_anchorings', 'payment_timeframe', 'registration_date', 'proforma_audit_status', 'renewal_score_at_release', 'proforma_approved_lists', 'tenure_years', 'prospect_renewal_date', 'starting_net', 'starting_vat', 'starting_gross', 'starting_membership_net', 'starting_package_net', 'starting_pqq_net', 'membership_net', 'package_net', 'pqqnet', 'total_amount', 'last_renewal', 'last_total_net_paid', 'last_connections', 'renewal_year', 'is_first_year', 'has_discount', 'discount_pct', 'price_increase_abs', 'price_increase_pct', 'price_increased_flag', 'band_ordinal', 'last_band_ordinal', 'band_changed', 'band_upgraded', 'band_downgraded', 'tenure_group_ordinal', 'anchor_group_ordinal', 'membership_status_ordinal', 'has_auto_renewal', 'has_world_pa

In [15]:
df.rename(columns={'high_risk_call_y': 'high_risk_call'}, inplace=True)
df.rename(columns={'dissatisfaction_issue_count_y': 'dissatisfaction_issue_count'}, inplace=True)


## Fill NaN from Unmatched Rows
Customers with no calls/emails in the window get NaN — fill with 0 since "no interaction" = 0.

In [16]:

renewal_cols = [c for c in df_renewal.columns if c != 'co_ref']
email_cols  = [c for c in df_email.columns if c != 'co_ref']
cc_cols   = [c for c in df_cc.columns if c != 'co_ref']
df[renewal_cols] = df[renewal_cols].fillna(0)
df[email_cols]  = df[email_cols].fillna(0)
df[cc_cols]    = df[cc_cols].fillna(0)

display(df)

,co_ref,renewal_month,sustainability_score,total_renewal_score_new,last_years_price,auto_renewal_score,status_scores,anchoring_score,tenure_scores,proforma_auto_renewal,proforma_world_pay_token,proforma_date,current_anchorings,payment_timeframe,registration_date,proforma_audit_status,renewal_score_at_release,proforma_approved_lists,tenure_years,prospect_renewal_date,starting_net,starting_vat,starting_gross,starting_membership_net,starting_package_net,starting_pqq_net,membership_net,package_net,pqqnet,total_amount,last_renewal,last_total_net_paid,last_connections,renewal_year,is_first_year,has_discount,discount_pct,price_increase_abs,price_increase_pct,price_increased_flag,band_ordinal,last_band_ordinal,band_changed,band_upgraded,band_downgraded,tenure_group_ordinal,anchor_group_ordinal,membership_status_ordinal,has_auto_renewal,has_world_pay_token,payment_method_bacs,payment_method_card,payment_method_cheque,payment_method_unknown,payment_method_world_pay,proforma_account_stage_membership_only,proforma_account_stage_published,proforma_account_stage_renewal_process,proforma_account_stage_retired,proforma_account_stage_suspended,proforma_account_stage_unknown,proforma_account_stage_vetting,churn,rc_14d_total_calls,rc_14d_inbound_calls,rc_14d_max_call_number,membership_renewal_decision,desire_to_cancel_encoded,customer_response_encoded,serious_complaint,other_complaint,complaint_count,discussion_on_price_increase,renewal_impact_due_to_price_increase,discount_or_waiver_requested,discount_offered,price_pressure_flag,price_range_discussed_flag,percentage_price_increase_mentioned,monetary_price_increase_mentioned,explicit_competitor_mention,explicit_switching_intent,mentioned_competitors,price_switching_mentioned,competitor_better_value,competitor_similar_value,competitor_better_service,competitor_cheaper,competitor_signal_count,competitor_threat_flag,agent_renewal_initiation,agent_flagged_membership_status_alert,agent_active_retention,call_reschedule_request,topic_by_customer,topic_by_agent,customer_asked_for_justification,high_risk_call_x,rc_14d_outbound_calls,email_count,crm_sentiment_encoded,crm_dissatisfied_flag,crm_satisfied_flag,crm_contractor_sentiment_score,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_competitors_mentioned,crm_platform_issues_raised,crm_membership_overdue,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,crm_dts_or_ssip_mentioned,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_contractor_engagement,crm_customer_payment_intention,crm_agent_chased_contractor,crm_accreditation_issues,crm_agent_chase_count,crm_auto_renewal_status,crm_membership_level_ordinal,high_risk_email,dissatisfaction_issue_count_x,engagement_score,sentiment_score_valid,cc_total_calls,cc_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,cc_urgency_getting_on_site,cc_chasing_response,cc_questionnaire_completion,cc_care_package_discussed,cc_external_consultant,cc_agent_cross_sell_attempt,cc_issues_within_questionnaire,sentiment_improved,sentiment_worsened,sentiment_change,high_risk_call,dissatisfaction_issue_count,pricing_pressure_flag,cc_care_package_assisted,cc_care_package_express,cc_care_package_not_discussed,cc_care_package_premier,cc_care_package_standard,cc_care_package_unknown,cc_call_initiated_by_agent,cc_call_initiated_by_customer,cc_call_initiated_by_not_relevant,cc_call_initiated_by_un

## Adding "Has Interaction" Flags

In [17]:
df['has_renewal_call'] = (df['rc_14d_total_calls'] > 0).astype(int)
df['has_email']   = (df['email_count'] > 0).astype(int)
df['has_cc_call'] = (df['cc_total_calls'] > 0).astype(int)

C:\Users\Bacon\AppData\Local\Temp\ipykernel_15184\820585843.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_renewal_call'] = (df['rc_14d_total_calls'] > 0).astype(int)
C:\Users\Bacon\AppData\Local\Temp\ipykernel_15184\820585843.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['has_email']   = (df['email_count'] > 0).astype(int)
C:\Users\Bacon\AppData\Local\Temp\ipykernel_15184\820585843.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

## Dropping co_ref and Remaining Non-Feature Columns


In [18]:
drop_cols = [
    'co_ref',           # ID — not a feature
    'renewal_month',    # raw date — not usable by model directly
]

df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(f"Shape after dropping ID cols: {df.shape}")

Shape after dropping ID cols: (113769, 175)


## Exporting final dataset

In [19]:
df.to_csv('../../../data/processed/final_dataset.csv', index=False)